<a href="https://colab.research.google.com/github/jawad66108/flyrank_ML_internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad66108/flyrank_ML_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip -q install duckdb huggingface_hub

In [5]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass(
    'Paste your Hugging Face READ token: '
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected successfully")

Paste your Hugging Face READ token: ··········
Connected successfully


### Unit of analysis + time window

One row represents one content item (`content_hash_id`) for one client (`client_hash_id`) over a daily reporting date (`report_date`).

For this contract, I will analyze content performance during a mid-panel month (March 2026) to avoid using the final sealed test month.

The goal is to understand and predict whether a content item is likely to experience an impressions decline based on historical search performance signals.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS rows_per_day
FROM {TABLES['fact_daily']}
WHERE report_date >= '2026-03-01'
AND report_date < '2026-04-01'
GROUP BY 1,2,3
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,rows_per_day
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,2026-03-01,1
1,client_62f4a7e64f5e0096,content_39d7361b4945d504,2026-03-01,1
2,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,2026-03-01,1
3,client_62f4a7e64f5e0096,content_4dc944b7d0b65ecc,2026-03-01,1
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,2026-03-01,1
5,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,2026-03-01,1
6,client_62f4a7e64f5e0096,content_7a7d3c7aa7cdfc5c,2026-03-01,1
7,client_62f4a7e64f5e0096,content_92c381fbd361212e,2026-03-01,1
8,client_62f4a7e64f5e0096,content_cdd114d71966c437,2026-03-01,1
9,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,2026-03-01,1


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT COUNT(*) AS rows
FROM {TABLES['fact_daily']}
WHERE report_date >= '2026-03-01'
AND report_date < '2026-04-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows
0,9841378


### Fields

#### Features
- gsc_impressions: search visibility before prediction.
- gsc_clicks: search traffic before prediction.
- gsc_avg_position: average ranking position before prediction.
- visible_queries: number of queries generating impressions.
- rare_share: share of impressions from rare queries.

#### Label
- is_declining: whether impressions decrease by more than 20% compared with the previous period.

#### Context
- client_hash_id: identifies the client group.
- content_hash_id: identifies the content item.
- report_date: reporting date.

#### Excluded
- URLs and client names are excluded because they are not needed for modeling and may contain identifying information.
- Future performance columns are excluded because they would create data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

This verifies that each row represents one content item for one client on one reporting date.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS duplicate_rows
FROM {TABLES['fact_daily']}
WHERE report_date >= '2026-03-01'
AND report_date < '2026-04-01'
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,duplicate_rows


In [9]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE report_date >= '2026-03-01'
AND report_date < '2026-04-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


This checks how many rows have valid availability flags.

In [11]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE report_date >= '2026-03-01'
AND report_date < '2026-04-01'
AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This verifies the size of my selected slice and confirms the available dates.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
### Data limits


### Data limits

This dataset cannot explain why performance changed because it does not contain external factors such as algorithm updates, content quality changes, competitors, or user intent changes.

The history is also unbalanced because different clients have different amounts of available data.

Some observations may have limited history, so features requiring long lookback windows may not exist for every content item.

The dataset supports directional prediction and decision support, not causal conclusions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.